In [1]:
library(ggplot2)
library(dplyr)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [2]:
setwd("/mnt/home3/miska/nm667/scratch/inProgress/mice_PiRNA/analysis/sRNA_deseq/sRNA_STAR_testing")

In [3]:
data <- read.csv("/mnt/home3/miska/nm667/scratch/inProgress/mice_PiRNA/analysis/sRNA_deseq/data/sRNA_STAR_testing.csv", header=TRUE,sep=';')
data

Strains,TimePoint,Mismatch,Multimap,AnchorMultimap,Replicate,Measure.Names,Measure.Values
<chr>,<chr>,<int>,<int>,<int>,<int>,<chr>,<dbl>
C57BL_6NJ,E16.5,0,50,100,3,Uniquely mapped reads,35.70
C57BL_6NJ,E16.5,0,50,100,2,Uniquely mapped reads,34.61
C57BL_6NJ,E16.5,0,50,100,1,Uniquely mapped reads,38.43
C57BL_6NJ,E16.5,0,50,100,3,Reads Mapped To Multiple Loci,26.99
C57BL_6NJ,E16.5,0,50,100,2,Reads Mapped To Multiple Loci,25.89
C57BL_6NJ,E16.5,0,50,100,1,Reads Mapped To Multiple Loci,25.11
C57BL_6NJ,E16.5,0,50,100,3,Reads Mapped To Too Many Loci,3.43
C57BL_6NJ,E16.5,0,50,100,2,Reads Mapped To Too Many Loci,3.47
C57BL_6NJ,E16.5,0,50,100,1,Reads Mapped To Too Many Loci,3.52


In [4]:
summary_data <- data %>%
  group_by(Mismatch, Multimap, AnchorMultimap, Measure.Names) %>%
  summarise(Mean = mean(Measure.Values),
            SE = sd(Measure.Values)/sqrt(n()))

`summarise()` has grouped output by 'Mismatch', 'Multimap', 'AnchorMultimap'.
You can override using the `.groups` argument.


In [5]:
summary_data <- summary_data %>%
  arrange(Mismatch, Multimap, AnchorMultimap, Measure.Names) %>% # Ensuring data is in correct order
  group_by(Mismatch, Multimap, AnchorMultimap) %>%
  mutate(cumulativeMean = cumsum(Mean))


In [6]:
# Define custom colors for stacked bars and error bars
stacked_bar_colors <- c( "#1f77b4","#2ca02c","#ff7f0e", "#d62728" )  # Replace with your colors
error_bar_color <- "#000000"   # Replace with your color


plot <- ggplot(summary_data, aes(x=Measure.Names, y=Mean, fill=Measure.Names)) +
  geom_errorbar(aes(ymax = cumulativeMean + SE, ymin = cumulativeMean - SE ), position="stack", width=4) +
  geom_bar(stat="identity", position="stack",width=10) +
  labs(x=NULL, y="Value") +  # You can also set the x label to NULL
  facet_grid(Measure.Names ~  Mismatch + Multimap + AnchorMultimap  ) +
  scale_fill_manual(values = stacked_bar_colors) +  # Set custom fill colors  
  theme_minimal() +
  theme(strip.text.x = element_text(angle = 90, hjust = 1),
        axis.title.x=element_blank(), 
        axis.text.x=element_blank())

ggsave(filename = "sRNA_STAR_testing.pdf", plot = plot, width = 7, height = 7)



Warning message:
“Stacking not well defined when not anchored on the axis”


In [1]:
run_data <- read.csv("/mnt/home3/miska/nm667/scratch/inProgress/mice_PiRNA/analysis/sRNA_deseq/data/sRNA_testing_runTime.csv", header=TRUE,sep=';')
run_data

Finished,Started,Mismatch,Multimap,Replicate,TimePoint,AnchorMultimap
<chr>,<chr>,<int>,<int>,<int>,<chr>,<int>
Apr 09 13:18:44,Apr 09 12:22:52,1,50,1,E16.5,100
Apr 09 13:29:36,Apr 09 12:34:07,1,50,2,E16.5,100
Apr 09 13:50:19,Apr 09 12:35:01,1,50,3,E16.5,100
Apr 09 16:13:49,Apr 09 14:55:25,10,200,1,E16.5,400
Apr 09 16:15:35,Apr 09 14:57:32,10,200,2,E16.5,400
Apr 09 18:01:16,Apr 09 15:04:10,10,200,3,E16.5,400
Apr 09 21:04:25,Apr 09 19:49:24,0,200,2,E16.5,400
Apr 09 21:09:54,Apr 09 19:44:00,0,200,1,E16.5,400
Apr 09 22:05:56,Apr 09 20:01:19,0,200,3,E16.5,400


In [3]:
library(ggplot2)
library(dplyr)



# Calculate time difference in minutes
run_data$Runtime <- as.numeric(difftime(as.POSIXct(run_data$Finished, format="%b %d %H:%M:%S"), 
                                    as.POSIXct(run_data$Started, format="%b %d %H:%M:%S"), units = "mins"))

                                    
# Summarize data
summary_data <- run_data %>% 
  group_by(Mismatch, Multimap, AnchorMultimap) %>% 
  summarize(Mean_Runtime = mean(Runtime), SD_Runtime = sd(Runtime))

# Plotting
plot <- ggplot(summary_data, aes(x=interaction(Mismatch, Multimap,  AnchorMultimap), y=Mean_Runtime)) +
  geom_errorbar(aes(ymin=Mean_Runtime-SD_Runtime, ymax=Mean_Runtime+SD_Runtime), width=.2) +
  geom_bar(stat="identity",width=10) +  theme_minimal()  +
  theme(axis.text.x = NULL) +
   # You can also set the x label to NULL
  facet_grid( ~  Mismatch + Multimap + AnchorMultimap  ) + 
  labs(x= NULL, y='Runtime (minutes)', title='Runtime with Error Bars') 
# Save plot with specific dimensions
ggsave(filename = "sRNA_STAR_testing_runtime.pdf",  plot, width=7, height=3)

`summarise()` has grouped output by 'Mismatch', 'Multimap'. You can override
using the `.groups` argument.
